
spark automatically enable broadcast if table size is less than 10MB
for table size greater than 10MB we have to force broadcast join

In [0]:
# CREATE LARGE DATA TABLE - Large Table (Orders → Millions of rows)

from pyspark.sql.functions import rand

orders_df = spark.range(0, 200000000) \
    .withColumn("customer_id", (rand()*5000).cast("int")) \
    .withColumn("amount", (rand()*1000).cast("int"))

# CREATE SMALL DATA TABLE - Small Table (Customers → 5000 rows) greater than 10MB

from pyspark.sql.functions import lit

customers_df = spark.range(0, 500000) \
    .withColumnRenamed("id", "customer_id") \
    .withColumn("customer_name", lit("customer_" * 50))   # increase size

# Step 4: Verify Table Size (>10MB)

from pyspark.sql.functions import length, sum as _sum

size_df = customers_df.select(
    _sum(length("customer_name")).alias("total_bytes")
)

size_bytes = size_df.collect()[0]["total_bytes"]

print("Size in MB:", size_bytes / (1024 * 1024))

In [0]:
# (Better Control) Disable Auto Broadcast Completely

# 👉 This is important for clean testing

# spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)



# Performance Comparison

import time

# Without Broadcast
start = time.time()
orders_df.join(customers_df, "customer_id").count()
print("No Broadcast:", time.time() - start)

# With Broadcast
start = time.time()
orders_df.join(broadcast(customers_df), "customer_id").count()
print("Forced Broadcast:", time.time() - start)